In [2]:
# ============================================================
# IMPORTS : on charge toutes les librairies du projet
# ============================================================
import pandas as pd                # tableaux de données (DataFrame)
import numpy as np                 # calculs numériques
import matplotlib.pyplot as plt    # graphiques de base
import seaborn as sns              # graphiques statistiques plus jolis
import sklearn                     # machine learning
import imblearn                    # gestion du déséquilibre des classes
import shap                        # interprétation du modèle

# On affiche les versions : preuve que l'environnement fonctionne
print("pandas :", pd.__version__)
print("numpy :", np.__version__)
print("seaborn :", sns.__version__)
print("scikit-learn :", sklearn.__version__)
print("imbalanced-learn :", imblearn.__version__)
print("shap :", shap.__version__)

# Petit test : on vérifie qu'on arrive bien à lire les 3 fichiers
for fichier in ["extrait_sirh.csv", "extrait_eval.csv", "extrait_sondage.csv"]:
    df_test = pd.read_csv(f"data/{fichier}")
    print(f"{fichier} : {df_test.shape[0]} lignes, {df_test.shape[1]} colonnes")

pandas : 3.0.6
numpy : 2.5.3
seaborn : 0.13.2
scikit-learn : 1.9.1
imbalanced-learn : 0.14.2
shap : 0.52.0
extrait_sirh.csv : 1470 lignes, 12 colonnes
extrait_eval.csv : 1470 lignes, 10 colonnes
extrait_sondage.csv : 1470 lignes, 12 colonnes


d:\GitProjets\Projet4\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Projet 4 – Analyse des causes de démission chez TechNova Partners
## 1. Chargement et découverte des données

In [ ]:
# Chaque fichier vient d'un système différent de TechNova :
#  - SIRH    : infos RH (poste, âge, salaire, ancienneté...)
#  - EVAL    : évaluations annuelles et satisfaction
#  - SONDAGE : sondage bien-être + la cible "a quitté l'entreprise"
df_sirh = pd.read_csv("data/extrait_sirh.csv")
df_eval = pd.read_csv("data/extrait_eval.csv")
df_sondage = pd.read_csv("data/extrait_sondage.csv")

### 1.1 Découverte du fichier SIRH

In [18]:
# 1) Les 5 premières lignes
display(df_sirh.head())

# 2) Le type de chaque colonne et le nombre de valeurs non vides
df_sirh.info()

# 3) Les statistiques des colonnes numériques (moyenne, min, max...)
display(df_sirh.describe())

# 4) Le nombre de valeurs différentes par colonne
display(df_sirh.nunique())

,id_employee,age,genre,revenu_mensuel,statut_marital,departement,poste,nombre_experiences_precedentes,nombre_heures_travailless,annee_experience_totale,annees_dans_l_entreprise,annees_dans_le_poste_actuel
0,1,41,F,5993,Célibataire,Commercial,Cadre Commercial,8,80,8,6,4
1,2,49,M,5130,Marié(e),Consulting,Assistant de Direction,1,80,10,10,7
2,4,37,M,2090,Célibataire,Consulting,Consultant,6,80,7,0,0
3,5,33,F,2909,Marié(e),Consulting,Assistant de Direction,1,80,8,8,7
4,7,27,M,3468,Marié(e),Consulting,Consultant,9,80,6,2,2


<class 'pandas.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 12 columns):
 #   Column                          Non-Null Count  Dtype
---  ------                          --------------  -----
 0   id_employee                     1470 non-null   int64
 1   age                             1470 non-null   int64
 2   genre                           1470 non-null   str  
 3   revenu_mensuel                  1470 non-null   int64
 4   statut_marital                  1470 non-null   str  
 5   departement                     1470 non-null   str  
 6   poste                           1470 non-null   str  
 7   nombre_experiences_precedentes  1470 non-null   int64
 8   nombre_heures_travailless       1470 non-null   int64
 9   annee_experience_totale         1470 non-null   int64
 10  annees_dans_l_entreprise        1470 non-null   int64
 11  annees_dans_le_poste_actuel     1470 non-null   int64
dtypes: int64(8), str(4)
memory usage: 137.9 KB


,id_employee,age,revenu_mensuel,nombre_experiences_precedentes,nombre_heures_travailless,annee_experience_totale,annees_dans_l_entreprise,annees_dans_le_poste_actuel
count,1470.000000,1470.000000,1470.000000,1470.000000,1470.0,1470.000000,1470.000000,1470.000000
mean,1024.865306,36.923810,6502.931293,2.693197,80.0,11.279592,7.008163,4.229252
std,602.024335,9.135373,4707.956783,2.498009,0.0,7.780782,6.126525,3.623137
min,1.000000,18.000000,1009.000000,0.000000,80.0,0.000000,0.000000,0.000000
25%,491.250000,30.000000,2911.000000,1.000000,80.0,6.000000,3.000000,2.000000
50%,1020.500000,36.000000,4919.000000,2.000000,80.0,10.000000,5.000000,3.000000
75%,1555.750000,43.000000,8379.000000,4.000000,80.0,15.000000,9.000000,7.000000
max,2068.000000,60.000000,19999.000000,9.000000,80.0,40.000000,40.000000,18.000000


id_employee                       1470
age                                 43
genre                                2
revenu_mensuel                    1349
statut_marital                       3
departement                          3
poste                                9
nombre_experiences_precedentes      10
nombre_heures_travailless            1
annee_experience_totale             40
annees_dans_l_entreprise            37
annees_dans_le_poste_actuel         19
dtype: int64

#### **Observations fichier SIRH**
- Pas de valeurs manquantes ni de doublons
- `nombre_heures_travailless` vaut 80 pour tout le monde,colonne inutile à supprimer
- Salaire max à 19 999 € : valeur très haute, à vérifier

In [19]:
# Vérification du salaire max à 19999
# Salaire min / médian / max par poste
# A priori, pas une abbération
display(df_sirh.groupby("poste")["revenu_mensuel"].agg(["min", "median", "max"]).sort_values("median"))

,min,median,max
poste,,,
Représentant Commercial,1052,2579.0,6632
Consultant,1102,2886.0,7403
Assistant de Direction,1009,2887.5,9724
Ressources Humaines,1555,3093.0,10725
Cadre Commercial,4001,6231.0,13872
Tech Lead,4011,6447.0,13973
Manager,4000,6811.0,13966
Directeur Technique,11031,16510.0,19973
Senior Manager,11244,17454.5,19999


### 1.2 Découverte du fichier EVAL

In [23]:
# 1) Les 5 premières lignes
display(df_eval.head())

# 2) Le type de chaque colonne et le nombre de valeurs non vides
df_eval.info()

# 3) Les statistiques des colonnes numériques (moyenne, min, max...)
display(df_eval.describe())

# 4) Le nombre de valeurs différentes par colonne
display(df_eval.nunique())

,satisfaction_employee_environnement,note_evaluation_precedente,niveau_hierarchique_poste,satisfaction_employee_nature_travail,satisfaction_employee_equipe,satisfaction_employee_equilibre_pro_perso,eval_number,note_evaluation_actuelle,heure_supplementaires,augementation_salaire_precedente
0,2,3,2,4,1,1,E_1,3,Oui,11 %
1,3,2,2,2,4,3,E_2,4,Non,23 %
2,4,2,1,3,2,3,E_4,3,Oui,15 %
3,4,3,1,3,3,3,E_5,3,Oui,11 %
4,1,3,1,2,4,3,E_7,3,Non,12 %


<class 'pandas.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 10 columns):
 #   Column                                     Non-Null Count  Dtype
---  ------                                     --------------  -----
 0   satisfaction_employee_environnement        1470 non-null   int64
 1   note_evaluation_precedente                 1470 non-null   int64
 2   niveau_hierarchique_poste                  1470 non-null   int64
 3   satisfaction_employee_nature_travail       1470 non-null   int64
 4   satisfaction_employee_equipe               1470 non-null   int64
 5   satisfaction_employee_equilibre_pro_perso  1470 non-null   int64
 6   eval_number                                1470 non-null   str  
 7   note_evaluation_actuelle                   1470 non-null   int64
 8   heure_supplementaires                      1470 non-null   str  
 9   augementation_salaire_precedente           1470 non-null   str  
dtypes: int64(7), str(3)
memory usage: 115.0 KB


,satisfaction_employee_environnement,note_evaluation_precedente,niveau_hierarchique_poste,satisfaction_employee_nature_travail,satisfaction_employee_equipe,satisfaction_employee_equilibre_pro_perso,note_evaluation_actuelle
count,1470.000000,1470.000000,1470.000000,1470.000000,1470.000000,1470.000000,1470.000000
mean,2.721769,2.729932,2.063946,2.728571,2.712245,2.761224,3.153741
std,1.093082,0.711561,1.106940,1.102846,1.081209,0.706476,0.360824
min,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,3.000000
25%,2.000000,2.000000,1.000000,2.000000,2.000000,2.000000,3.000000
50%,3.000000,3.000000,2.000000,3.000000,3.000000,3.000000,3.000000
75%,4.000000,3.000000,3.000000,4.000000,4.000000,3.000000,3.000000
max,4.000000,4.000000,5.000000,4.000000,4.000000,4.000000,4.000000


satisfaction_employee_environnement             4
note_evaluation_precedente                      4
niveau_hierarchique_poste                       5
satisfaction_employee_nature_travail            4
satisfaction_employee_equipe                    4
satisfaction_employee_equilibre_pro_perso       4
eval_number                                  1470
note_evaluation_actuelle                        2
heure_supplementaires                           2
augementation_salaire_precedente               15
dtype: int64

#### **Observations fichier EVAL**
- Pas de valeurs manquantes ni de doublons
- `augementation_salaire_precedente` : nombre stocké en texte à convertir (+ faute dans le nom colonne)
- `heure_supplementaires` : Oui/Non à convertir en binaire
- `eval_number` : identifiant au format "E_1" bonne piste pour jointure

### 1.3 Découverte du fichier SONDAGE

In [21]:
# 1) Les 5 premières lignes
display(df_sondage.head())

# 2) Le type de chaque colonne et le nombre de valeurs non vides
df_sondage.info()

# 3) Les statistiques des colonnes numériques (moyenne, min, max...)
display(df_sondage.describe())

# 4) Le nombre de valeurs différentes par colonne
display(df_sondage.nunique())

,a_quitte_l_entreprise,nombre_participation_pee,nb_formations_suivies,nombre_employee_sous_responsabilite,code_sondage,distance_domicile_travail,niveau_education,domaine_etude,ayant_enfants,frequence_deplacement,annees_depuis_la_derniere_promotion,annes_sous_responsable_actuel
0,Oui,0,0,1,1,1,2,Infra & Cloud,Y,Occasionnel,0,5
1,Non,1,3,1,2,8,1,Infra & Cloud,Y,Frequent,1,7
2,Oui,0,3,1,4,2,2,Autre,Y,Occasionnel,0,0
3,Non,0,3,1,5,3,4,Infra & Cloud,Y,Frequent,3,0
4,Non,1,3,1,7,2,1,Transformation Digitale,Y,Occasionnel,2,2


<class 'pandas.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 12 columns):
 #   Column                               Non-Null Count  Dtype
---  ------                               --------------  -----
 0   a_quitte_l_entreprise                1470 non-null   str  
 1   nombre_participation_pee             1470 non-null   int64
 2   nb_formations_suivies                1470 non-null   int64
 3   nombre_employee_sous_responsabilite  1470 non-null   int64
 4   code_sondage                         1470 non-null   int64
 5   distance_domicile_travail            1470 non-null   int64
 6   niveau_education                     1470 non-null   int64
 7   domaine_etude                        1470 non-null   str  
 8   ayant_enfants                        1470 non-null   str  
 9   frequence_deplacement                1470 non-null   str  
 10  annees_depuis_la_derniere_promotion  1470 non-null   int64
 11  annes_sous_responsable_actuel        1470 non-null   int64
dtypes: 

,nombre_participation_pee,nb_formations_suivies,nombre_employee_sous_responsabilite,code_sondage,distance_domicile_travail,niveau_education,annees_depuis_la_derniere_promotion,annes_sous_responsable_actuel
count,1470.000000,1470.000000,1470.0,1470.000000,1470.000000,1470.000000,1470.000000,1470.000000
mean,0.793878,2.799320,1.0,1024.865306,9.192517,2.912925,2.187755,4.123129
std,0.852077,1.289271,0.0,602.024335,8.106864,1.024165,3.222430,3.568136
min,0.000000,0.000000,1.0,1.000000,1.000000,1.000000,0.000000,0.000000
25%,0.000000,2.000000,1.0,491.250000,2.000000,2.000000,0.000000,2.000000
50%,1.000000,3.000000,1.0,1020.500000,7.000000,3.000000,1.000000,3.000000
75%,1.000000,3.000000,1.0,1555.750000,14.000000,4.000000,3.000000,7.000000
max,3.000000,6.000000,1.0,2068.000000,29.000000,5.000000,15.000000,17.000000


a_quitte_l_entreprise                     2
nombre_participation_pee                  4
nb_formations_suivies                     7
nombre_employee_sous_responsabilite       1
code_sondage                           1470
distance_domicile_travail                29
niveau_education                          5
domaine_etude                             6
ayant_enfants                             1
frequence_deplacement                     3
annees_depuis_la_derniere_promotion      16
annes_sous_responsable_actuel            18
dtype: int64

#### **Observations fichier SONDAGE**

- Pas de valeurs manquantes ni de doublons
- `a_quitte_l_entreprise` : **variable cible** (Oui/Non), à convertir en 1/0
- `code_sondage` : 1 470 valeurs uniques, semble correspondre à `id_employee` du SIRH, **à vérifier** avant la jointure
- Colonnes constantes (une seule valeur pour tous) donc à supprimer :
  - `nombre_employee_sous_responsabilite` (toujours 1)
  - `ayant_enfants` (toujours "Y")
- `annes_sous_responsable_actuel` : faute dans le nom, à voir si on corrige

**Pistes pour préparation modèle**
- `domaine_etude` : texte sans ordre → encodage one-hot
- `frequence_deplacement` : texte avec un ordre naturel (Aucun < Occasionnel < Fréquent), encodage à choisir
- `distance_domicile_travail` : découpage en tranches éventuel

In [25]:
# Les identifiants du sondage sont-ils exactement les mêmes que ceux du SIRH ?
# set() = ensemble de valeurs, l'ordre n'a pas d'importance
print(set(df_sondage["code_sondage"]) == set(df_sirh["id_employee"]))

True


Ce sont bien les mêmes valeurs que pour le fichier SIRH, cela servira de clé